## Experiment 4 — Hypothesis Testing
**Dataset:** Titanic Dataset (Passenger Survival)  
**Objective:** Apply inferential statistical tests to validate or reject null hypotheses about group differences and feature associations.  
**Techniques:** Welch's t-test, Chi-square test of independence, One-way ANOVA, Tukey HSD post-hoc test (α = 0.05).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, chi2_contingency, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

### 1. Setup & Imports
Import scipy (for t-test and chi-square), statsmodels (for ANOVA and Tukey HSD), and pandas. Set display options for clean output formatting.

In [ ]:
df = pd.read_csv("Titanic-Dataset.csv")
print("Dataset shape:", df.shape)

### 2. Load Dataset
Read the dataset. Hypothesis tests will be applied to specific features, so it is important to understand the schema and available group variables.

In [ ]:
alpha = 0.05
num_col, group_col = "Age", "Sex"
group_a, group_b = "male", "female"
cat_col1, cat_col2 = "Sex", "Survived"
anova_num_col, anova_group_col = "Fare", "Pclass"

def decide(p):
    return "Reject H0" if p < alpha else "Fail to reject H0"

print(df[group_col].value_counts())
print(df[anova_group_col].value_counts())

### 3. Welch's Two-Sample t-Test
Test whether the means of a continuous variable differ significantly between two independent groups. Welch's variant is used because it does not assume equal population variances — making it more robust than Student's t-test.  
**H₀:** The group means are equal.  
**H₁:** The group means are significantly different (α = 0.05).

In [ ]:
g1 = df.loc[df[group_col] == group_a, num_col].dropna()
g2 = df.loc[df[group_col] == group_b, num_col].dropna()
print(f"{group_a} count: {len(g1)}, mean: {g1.mean():.2f}, std: {g1.std():.2f}")
print(f"{group_b} count: {len(g2)}, mean: {g2.mean():.2f}, std: {g2.std():.2f}")

plt.figure(figsize=(6, 4))
sns.boxplot(data=df[df[group_col].isin([group_a, group_b])], x=group_col, y=num_col)
plt.title(f"Comparison of {num_col} by {group_col}")
plt.tight_layout()
plt.show()

t_stat, t_p = ttest_ind(g1, g2, equal_var=False)
print(f"Welch's t-test: Statistic = {t_stat:.4f}, p-value = {t_p:.4e} -> {decide(t_p)}")

### 4. Chi-Square Test of Independence
Test whether two categorical variables are statistically independent. A low p-value indicates a significant association between the variables.  
**H₀:** The two categorical variables are independent.  
**H₁:** There is a significant association between them (α = 0.05).

In [ ]:
chi_df = df[[cat_col1, cat_col2]].dropna()
observed = pd.crosstab(chi_df[cat_col1], chi_df[cat_col2])

plt.figure(figsize=(6, 4))
sns.heatmap(observed, annot=True, fmt="d", cmap="Blues")
plt.title(f"Contingency Table ({cat_col1} vs {cat_col2})")
plt.tight_layout()
plt.show()

chi2, chi_p, dof, expected = chi2_contingency(observed)
print(f"Chi-square test: Statistic = {chi2:.4f}, dof = {dof}, p-value = {chi_p:.4e} -> {decide(chi_p)}")

expected_df = pd.DataFrame(expected, index=observed.index, columns=observed.columns)
print("Expected Frequencies:\n", expected_df.round(2))

### 5. One-Way ANOVA
Test whether the means of a continuous variable differ significantly across three or more groups. ANOVA partitions the total variance into between-group and within-group components.  
**H₀:** All group means are equal.  
**H₁:** At least one group mean is significantly different (α = 0.05).

In [ ]:
groups_used = df[anova_group_col].value_counts().head(3).index.tolist()
anova_df = df[df[anova_group_col].isin(groups_used)]
samples = [anova_df.loc[anova_df[anova_group_col] == g, anova_num_col].dropna() for g in groups_used]

print(anova_df.groupby(anova_group_col)[anova_num_col].agg(["count", "mean"]))

plt.figure(figsize=(7, 4))
sns.boxplot(data=anova_df, x=anova_group_col, y=anova_num_col)
plt.title(f"{anova_num_col} across {anova_group_col}")
plt.tight_layout()
plt.show()

f_stat, anova_p = f_oneway(*samples)
print(f"One-way ANOVA: F-stat = {f_stat:.4f}, p-value = {anova_p:.4e} -> {decide(anova_p)}")

### 6. Tukey HSD Post-Hoc Test
When ANOVA is significant, Tukey's Honestly Significant Difference test identifies *which specific pairs* of groups differ. It controls the family-wise error rate across all pairwise comparisons, providing more rigour than running multiple t-tests.

In [ ]:
clean_anova = anova_df[[anova_num_col, anova_group_col]].dropna()
tukey = pairwise_tukeyhsd(clean_anova[anova_num_col], clean_anova[anova_group_col], alpha=alpha)
print(tukey)

### 7. Summary of Test Results
Collect all test statistics, p-values, and decisions (Reject / Fail to Reject H₀) into a summary DataFrame and export to CSV for reporting.

In [ ]:
results = pd.DataFrame({
    "Test": ["Welch's t-test", "Chi-square test", "One-way ANOVA"],
    "Statistic": [t_stat, chi2, f_stat],
    "p-value": [t_p, chi_p, anova_p]
})
results["Decision"] = results["p-value"].apply(decide)
print(results)
results.to_csv("hypothesis_testing_results.csv", index=False)